If your body is a system, protein kinases behave like tiny machines. There's a type of protein EGFR (Epidermal Growth Factor) that acts like a light switch that causes cells to grow; sometimes the switch gets stuck and leads to cancer.

In [ ]:
#pip install cptac

In [ ]:
#import libs
import pandas as pd
import cptac


In [26]:
mutations_WT = pd.read_csv('mutations.csv') 
# WT not needed since no EGFR mutations are present
mutation = mutations_WT[
    ~mutations_WT["EGFR"].str.contains("WT", case=False, na=False)
]

# EGFR = mutation
mutation = mutation.rename(columns={"EGFR": "mutation"})
mutation = mutation.rename(columns={"SAMPLE_ID": "Patient_ID"})
mutation["Patient_ID"] = mutation["Patient_ID"].str.replace(r"-T.*", "", regex=True)
mutation.head()

"""
#find a way to remove for loop (Markus dock points)
def classify_mutation(mutation):
    if pd.isna(mutation):
        return "Unknown"
    m = mutation.lower()
    if "l858r" in m:
        return "L858R"
    elif "del" in m or "e746" in m or "exon19" in m:
        return "Exon19"
    else:
        return "Other"
"""

def classify_column(df):
    m = df['mutation'].str.lower()
    
    df['classification'] = "Other"
    # mutation points for exon 19 and L858R
    df.loc[m.str.contains("del|e746|exon19", na=False), 'classification'] = "Exon19"
    df.loc[m.str.contains("l858r", na=False), 'classification'] = "L858R"
    df.loc[df['mutation'].isna(), 'classification'] = "Unknown"
    
    return df

egfr_group_counts = mutation['mutation'].value_counts()
print("Frequency of each EGFR group type:")
#display(egfr_group_counts)

mutation = classify_column(mutation)
mutation.head()

Frequency of each EGFR group type:


,Patient_ID,mutation,classification
3,TCGA-05-4382-01,R222L E545Q,Other
11,TCGA-05-4402-01,T751_I759delinsN I759N,Exon19
14,TCGA-05-4410-01,R377S,Other
28,TCGA-05-5423-01,L833F L861Q,Other
55,TCGA-17-Z026-01,G721V,Other


In [39]:
luad_data = cptac.Luad()
"""
Only need to be downloaded once 

download:
log2 ref normalized proteomic
log2 red tumor/normal proteomic
log2 ref normalized phosphoproteomic
log2 red tumor/normal phosphoproteomic
log2 ref normalized transcriptomic
log2 red tumor/normal transcriptomic
"""
proteomics = luad_data.get_proteomics(source='bcm')
phospho = luad_data.get_phosphoproteomics(source='bcm')
rna_seq = luad_data.get_transcriptomics(source='bcm')

#extract egfr data for kinase activity
target_sites = ['Y1068', 'Y1173'] #binding stie
egfr_rna = rna_seq[rna_seq.index == 'EGFR']
egfr_proteomics = proteomics[proteomics.index == 'EGFR']
egfr_phospho = phospho[phospho.index == 'EGFR']
site_level_phospho = egfr_phospho.filter(regex='Y1068|Y1173')
select_columns = [col for col in egfr_phospho.columns if any(site in col for site in target_sites)]

#mean from phoso data
egfr_activity = egfr_phospho[select_columns]
egfr_activity["EGFR_activity_mean"] = egfr_activity.mean(axis=1)

In [47]:
#combine mutations.csv and egfr_activity
cptac_df = pd.concat(
    [egfr_proteomics, egfr_activity, egfr_rna], axis=1)

# Flatten multi-level columns if they exist (gemini)
if isinstance(cptac_df.columns, pd.MultiIndex):
    cptac_df.columns = ['_'.join(col).strip('_') for col in cptac_df.columns.values]

cptac_df = cptac_df.reset_index() #make into columns
cptac_df = cptac_df.rename(columns={'index': 'Patient_ID'})

print("cptac_df shape before merge:", cptac_df.shape)
print("mutation shape before merge:", mutation.shape)

checkpoint_df = pd.merge(mutation, cptac_df, on='Patient_ID', how='inner')
print("checkpoint_df shape after merge:", checkpoint_df.shape)
checkpoint_df.head()

cptac_df shape before merge: (0, 71719)
mutation shape before merge: (70, 3)
checkpoint_df shape after merge: (0, 71721)


,Patient_ID,mutation,classification,A1BG_ENSG00000121410.12,A1CF_ENSG00000148584.15,A2M_ENSG00000175899.15,A2ML1_ENSG00000166535.20,A4GALT_ENSG00000128274.17,AAAS_ENSG00000094914.14,AACS_ENSG00000081760.17,...,ZXDB_ENSG00000198455.4,ZXDC_ENSG00000070476.15,ZYG11A_ENSG00000203995.10,ZYG11AP1_ENSG00000232242.2,ZYG11B_ENSG00000162378.13,ZYX_ENSG00000159840.16,ZYXP1_ENSG00000274572.1,ZZEF1_ENSG00000074755.15,hsa-mir-1253_ENSG00000272920.1,hsa-mir-423_ENSG00000266919.3


In [46]:
print("First 5 columns in cptac_df:", cptac_df.columns[:5].tolist())
print("cptac_df shape:", cptac_df.shape)
print("\nFirst few rows of cptac_df:")
print(cptac_df.iloc[:3, :5])

First 5 columns in cptac_df: ['Patient_ID', 'A1BG_ENSG00000121410.12', 'A1CF_ENSG00000148584.15', 'A2M_ENSG00000175899.15', 'A2ML1_ENSG00000166535.20']
cptac_df shape: (0, 71719)

First few rows of cptac_df:
Empty DataFrame
Columns: [Patient_ID, A1BG_ENSG00000121410.12, A1CF_ENSG00000148584.15, A2M_ENSG00000175899.15, A2ML1_ENSG00000166535.20]
Index: []


In [48]:
egfr_total_protein = proteomics['EGFR']
egfr_phospho = phospho['EGFR']
egfr_rna = rna_seq['EGFR']

target_sites = ['Y1068', 'Y1173'] #binding stie 
available_sites = [col for col in egfr_phospho.columns if any(site in col for site in target_sites)]
egfr_activity = egfr_phospho[available_sites]

In [49]:
# Save the final checkpoint dataframe to CSV
checkpoint_df.to_csv('checkpoint_3_final.csv', index=False)
print(f"Final CSV saved as 'checkpoint_3_final.csv'")
print(f"Shape: {checkpoint_df.shape}")
print(f"Columns: {checkpoint_df.columns.tolist()[:10]}...")  # Show first 10 columns

Final CSV saved as 'checkpoint_3_final.csv'
Shape: (0, 71721)
Columns: ['Patient_ID', 'mutation', 'classification', 'A1BG_ENSG00000121410.12', 'A1CF_ENSG00000148584.15', 'A2M_ENSG00000175899.15', 'A2ML1_ENSG00000166535.20', 'A4GALT_ENSG00000128274.17', 'AAAS_ENSG00000094914.14', 'AACS_ENSG00000081760.17']...


In [50]:
# Debug: Check Patient ID formats in both datasets
print("Patient IDs in mutation data:")
print(mutation['Patient_ID'].head(10).tolist())
print(f"\nTotal unique Patient IDs in mutation: {mutation['Patient_ID'].nunique()}")

print("\n" + "="*50)
print("Patient IDs in cptac_df:")
print(cptac_df['Patient_ID'].head(10).tolist() if len(cptac_df) > 0 else "cptac_df is empty")
print(f"Total unique Patient IDs in cptac_df: {cptac_df['Patient_ID'].nunique() if len(cptac_df) > 0 else 0}")

# Check if index (before reset) contains patient info
print("\n" + "="*50)
print("Index of egfr_proteomics (before reset):")
print(egfr_proteomics.index.tolist()[:5] if len(egfr_proteomics) > 0 else "Empty")

print("\nColumns in egfr_proteomics:")
print(egfr_proteomics.columns.tolist()[:5])

Patient IDs in mutation data:
['TCGA-05-4382-01', 'TCGA-05-4402-01', 'TCGA-05-4410-01', 'TCGA-05-5423-01', 'TCGA-17-Z026-01', 'TCGA-17-Z032-01', 'TCGA-17-Z047-01', 'TCGA-17-Z048-01', 'TCGA-38-4627-01', 'TCGA-38-4628-01']

Total unique Patient IDs in mutation: 70

Patient IDs in cptac_df:
cptac_df is empty
Total unique Patient IDs in cptac_df: 0

Index of egfr_proteomics (before reset):
Empty

Columns in egfr_proteomics:
[('A1BG', 'ENSG00000121410.12'), ('A1CF', 'ENSG00000148584.15'), ('A2M', 'ENSG00000175899.15'), ('A2ML1', 'ENSG00000166535.20'), ('A4GALT', 'ENSG00000128274.17')]


In [51]:
# Check the actual structure of proteomics data
print("Proteomics data structure:")
print(f"Shape: {proteomics.shape}")
print(f"Index (first 5 patients): {proteomics.index.tolist()[:5]}")
print(f"\nColumns (MultiIndex, first 5): {proteomics.columns[:5].tolist()}")

# Check if EGFR is in columns
egfr_cols = [col for col in proteomics.columns if 'EGFR' in str(col)]
print(f"\nColumns containing 'EGFR': {egfr_cols[:5] if egfr_cols else 'None found'}")

# Extract EGFR from columns instead of index
if egfr_cols:
    egfr_data = proteomics[egfr_cols]
    print(f"\nEGFR data shape: {egfr_data.shape}")
    print(f"EGFR data index (first 5 patients): {egfr_data.index.tolist()[:5]}")

Proteomics data structure:
Shape: (207, 12431)
Index (first 5 patients): ['C3L-00001', 'C3L-00009', 'C3L-00080', 'C3L-00083', 'C3L-00093']

Columns (MultiIndex, first 5): [('A1BG', 'ENSG00000121410.12'), ('A1CF', 'ENSG00000148584.15'), ('A2M', 'ENSG00000175899.15'), ('A2ML1', 'ENSG00000166535.20'), ('A4GALT', 'ENSG00000128274.17')]

Columns containing 'EGFR': [('EGFR', 'ENSG00000146648.18')]

EGFR data shape: (207, 1)
EGFR data index (first 5 patients): ['C3L-00001', 'C3L-00009', 'C3L-00080', 'C3L-00083', 'C3L-00093']


In [52]:
print("="*60)
print("OPTION 2: TCGA-CPTAC Mapping File")
print("="*60)

# Check if there's a mapping file or if we need to create one
# First, let's check the clinical data files
print("\nChecking checkpoint_3.csv:")
try:
    clinical_data = pd.read_csv('checkpoint_3.csv')
    print(f"Shape: {clinical_data.shape}")
    print(f"Columns: {clinical_data.columns.tolist()[:10]}")
    print(f"First few values:")
    print(clinical_data[['Sample ID', 'Patient ID']].head())
except Exception as e:
    print(f"Error reading checkpoint_3.csv: {e}")

print("\n" + "-"*60)
print("\nChecking check_EGFR.csv:")
try:
    egfr_clinical = pd.read_csv('check_EGFR.csv')
    print(f"Shape: {egfr_clinical.shape}")
    print(f"Columns: {egfr_clinical.columns.tolist()[:10]}")
except Exception as e:
    print(f"Error reading check_EGFR.csv: {e}")

OPTION 2: TCGA-CPTAC Mapping File

Checking checkpoint_3.csv:
Shape: (15, 65)
Columns: ['Sample ID', 'Patient ID', 'Study ID', 'Female', 'Male', 'Diagnosis Age', 'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'American Joint Committee on Cancer Publication Version Type', 'Aneuploidy Score', 'Buffa Hypoxia Score']
First few values:
         Sample ID    Patient ID
0  TCGA-05-4402-01  TCGA-05-4402
1  TCGA-38-6178-01  TCGA-38-6178
2  TCGA-44-A4SU-01  TCGA-44-A4SU
3  TCGA-49-4501-01  TCGA-49-4501
4  TCGA-50-6591-01  TCGA-50-6591

------------------------------------------------------------

Checking check_EGFR.csv:
Shape: (12, 58)
Columns: ['Sample ID', 'Patient ID', 'Diagnosis Age', 'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Aneuploidy Score', 'Buffa Hypoxia Score', 'Cancer Type', 'TCGA PanCanAtlas Cancer Type Acronym', 'Cancer Type Detailed', 'Last Communication Contact from Initial Pathologic Diagnosis Date']


In [1]:
print("\n" + "="*60)
print("OPTION 3: Use Existing TCGA Clinical Data")
print("="*60)

# Option 3: Use check_EGFR.csv which already has TCGA IDs + EGFR status
print("\nLoading check_EGFR.csv (already has TCGA IDs + EGFR data):")
try:
    tcga_egfr = pd.read_csv('mutations.csv')
    print(f"Shape: {tcga_egfr.shape}")
    print(f"\nColumns: {tcga_egfr.columns.tolist()[:15]}")
    print(f"\nFirst 3 rows:")
    print(tcga_egfr.head(3))
    
    # Merge with mutation data on Patient ID
    print(f"\n✓ This data has Patient_ID column and already contains:")
    print(f"  - Disease-specific Survival status (survival event)")
    print(f"  - Months of disease-specific survival (follow-up time)")
    print(f"  - Clinical features")
    
except Exception as e:
    print(f"Error: {e}")


OPTION 3: Use Existing TCGA Clinical Data

Loading check_EGFR.csv (already has TCGA IDs + EGFR data):
Error: name 'pd' is not defined


In [56]:
# OPTION 3 IMPLEMENTATION: Merge mutations with existing clinical EGFR data
print("\nOPTION 3 IMPLEMENTATION:")
print("-" * 60)

try:
    # Load the existing TCGA clinical data that has EGFR info
    tcga_egfr = pd.read_csv('check_EGFR.csv')
    
    # Rename 'Patient ID' to 'Patient_ID' for consistency
    tcga_egfr = tcga_egfr.rename(columns={'Patient ID': 'Patient_ID'})
    
    # Prepare the mutation data (already has Patient_ID from mutations.csv)
    mutation_clean = mutation[['Patient_ID', 'mutation', 'classification']].copy()
    
    # Merge on Patient_ID
    final_data = pd.merge(mutation_clean, tcga_egfr, on='Patient_ID', how='inner')
    
    print(f"✓ Successfully merged!")
    print(f"Shape: {final_data.shape}")
    print(f"Rows: {final_data.shape[0]} patients with both mutation and clinical data")
    
    # Save this final dataset
    final_data.to_csv('checkpoint_3_option3_final.csv', index=False)
    print(f"\n✓ Saved to 'checkpoint_3_option3_final.csv'")
    
    print(f"\nColumns available ({len(final_data.columns)} total):")
    print(final_data.columns.tolist()[:20])
    
    print(f"\nFirst 3 rows:")
    print(final_data[['Patient_ID', 'mutation', 'classification', 'Disease-specific Survival status', 'Months of disease-specific survival']].head(3))
    
except Exception as e:
    print(f"Error in Option 3: {e}")
    import traceback
    traceback.print_exc()


OPTION 3 IMPLEMENTATION:
------------------------------------------------------------
✓ Successfully merged!
Shape: (0, 60)
Rows: 0 patients with both mutation and clinical data

✓ Saved to 'checkpoint_3_option3_final.csv'

Columns available (60 total):
['Patient_ID', 'mutation', 'classification', 'Sample ID', 'Diagnosis Age', 'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Aneuploidy Score', 'Buffa Hypoxia Score', 'Cancer Type', 'TCGA PanCanAtlas Cancer Type Acronym', 'Cancer Type Detailed', 'Last Communication Contact from Initial Pathologic Diagnosis Date', 'Birth from Initial Pathologic Diagnosis Date', 'Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value', 'Disease Free (Months)', 'Disease Free Status', 'Months of disease-specific survival', 'Disease-specific Survival status', 'Ethnicity Category', 'Form completion date']

First 3 rows:
Empty DataFrame
Columns: [Patient_ID, mutation, classification, Disease-specific Survival status, Months of

In [57]:
# Debug: Check why merge returned 0 rows
print("\nDEBUGGING MERGE:")
print("="*60)

tcga_egfr = pd.read_csv('check_EGFR.csv')
tcga_egfr = tcga_egfr.rename(columns={'Patient ID': 'Patient_ID'})

print(f"Patient_ID in mutation data:")
print(mutation['Patient_ID'].head(10).tolist())
print(f"Unique count: {mutation['Patient_ID'].nunique()}")

print(f"\nPatient_ID in tcga_egfr:")
print(tcga_egfr['Patient_ID'].head(10).tolist())
print(f"Unique count: {tcga_egfr['Patient_ID'].nunique()}")

# Check for intersecting Patient IDs
common = set(mutation['Patient_ID']).intersection(set(tcga_egfr['Patient_ID']))
print(f"\nCommon Patient IDs: {len(common)}")
if common:
    print(f"Sample: {list(common)[:5]}")
else:
    print("No overlap found - checking format differences...")
    print(f"\nMutation Patient ID format: {mutation['Patient_ID'].iloc[0]}")
    print(f"TCGA EGFR Patient ID format: {tcga_egfr['Patient_ID'].iloc[0]}")


DEBUGGING MERGE:
Patient_ID in mutation data:
['TCGA-05-4382-01', 'TCGA-05-4402-01', 'TCGA-05-4410-01', 'TCGA-05-5423-01', 'TCGA-17-Z026-01', 'TCGA-17-Z032-01', 'TCGA-17-Z047-01', 'TCGA-17-Z048-01', 'TCGA-38-4627-01', 'TCGA-38-4628-01']
Unique count: 70

Patient_ID in tcga_egfr:
['TCGA-05-4402', 'TCGA-38-6178', 'TCGA-44-A4SU', 'TCGA-49-4501', 'TCGA-50-6591', 'TCGA-55-6981', 'TCGA-55-8096', 'TCGA-55-A48Z', 'TCGA-69-7760', 'TCGA-86-8055']
Unique count: 12

Common Patient IDs: 0
No overlap found - checking format differences...

Mutation Patient ID format: TCGA-05-4382-01
TCGA EGFR Patient ID format: TCGA-05-4402


In [58]:
# FIX: Normalize Patient_ID by removing the sample suffix (-01, -02, etc.)
print("\nFIXING PATIENT ID FORMAT:")
print("="*60)

# Remove the sample suffix from mutation data (last -XX part)
mutation['Patient_ID'] = mutation['Patient_ID'].str.replace(r'-\d{2}$', '', regex=True)

print(f"Mutation Patient_ID after normalization:")
print(mutation['Patient_ID'].head(10).tolist())

# Now merge again with tcga_egfr
mutation_clean = mutation[['Patient_ID', 'mutation', 'classification']].copy()

common = set(mutation_clean['Patient_ID']).intersection(set(tcga_egfr['Patient_ID']))
print(f"\n✓ Common Patient IDs after normalization: {len(common)}")
print(f"Sample: {list(common)[:5]}")

# Merge on Patient_ID
final_data = pd.merge(mutation_clean, tcga_egfr, on='Patient_ID', how='inner')

print(f"\n✓ Successfully merged!")
print(f"Final merged dataset shape: {final_data.shape}")
print(f"Rows: {final_data.shape[0]} patients with mutation and clinical data")

# Save this final dataset
final_data.to_csv('checkpoint_3_final_merged.csv', index=False)
print(f"\n✓ Saved to 'checkpoint_3_final_merged.csv'")

print(f"\nFinal dataset columns ({len(final_data.columns)} total):")
print(final_data.columns.tolist()[:15])

print(f"\nFirst 3 rows of final data:")
display(final_data[['Patient_ID', 'mutation', 'classification', 'Disease-specific Survival status', 'Months of disease-specific survival']].head(3))


FIXING PATIENT ID FORMAT:
Mutation Patient_ID after normalization:
['TCGA-05-4382', 'TCGA-05-4402', 'TCGA-05-4410', 'TCGA-05-5423', 'TCGA-17-Z026', 'TCGA-17-Z032', 'TCGA-17-Z047', 'TCGA-17-Z048', 'TCGA-38-4627', 'TCGA-38-4628']

✓ Common Patient IDs after normalization: 12
Sample: ['TCGA-38-6178', 'TCGA-MP-A4T9', 'TCGA-55-6981', 'TCGA-50-6591', 'TCGA-86-8055']

✓ Successfully merged!
Final merged dataset shape: (12, 60)
Rows: 12 patients with mutation and clinical data

✓ Saved to 'checkpoint_3_final_merged.csv'

Final dataset columns (60 total):
['Patient_ID', 'mutation', 'classification', 'Sample ID', 'Diagnosis Age', 'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Aneuploidy Score', 'Buffa Hypoxia Score', 'Cancer Type', 'TCGA PanCanAtlas Cancer Type Acronym', 'Cancer Type Detailed', 'Last Communication Contact from Initial Pathologic Diagnosis Date', 'Birth from Initial Pathologic Diagnosis Date', 'Last Alive Less Initial Pathologic Diagnosis Date Calculated Day 

,Patient_ID,mutation,classification,Disease-specific Survival status,Months of disease-specific survival
0,TCGA-05-4402,T751_I759delinsN I759N,Exon19,0.0,8.021830
1,TCGA-38-6178,E746_A750del,Exon19,0.0,14.728606
2,TCGA-44-A4SU,G719C S768I,Other,1.0,13.446428


In [54]:
# OPTION 2: Create a TCGA-CPTAC mapping file from available data
print("\n" + "="*60)
print("OPTION 2 IMPLEMENTATION: Create TCGA-CPTAC Mapping File")
print("="*60)

print("\nNote: TCGA and CPTAC are different cohorts with different patient IDs")
print("A mapping file would need to be obtained from CPTAC documentation or")
print("created if the datasets share any overlapping patients.\n")

# Check if there's any way to match them
print("Checking if there's overlap between TCGA and CPTAC patient IDs...")

tcga_ids = set(mutation['Patient_ID'].unique())
cptac_ids = set(proteomics.index.tolist())

print(f"TCGA IDs sample: {list(tcga_ids)[:3]}")
print(f"CPTAC IDs sample: {list(cptac_ids)[:3]}")

overlap = tcga_ids.intersection(cptac_ids)
print(f"\nPatient IDs in common: {len(overlap)}")

if len(overlap) == 0:
    print("\n❌ No overlapping patient IDs found between:")
    print("   - TCGA mutation data (TCGA-XX-XXXX format)")
    print("   - CPTAC proteomics data (C3L-XXXXX format)")
    print("\n✓ RECOMMENDATION: Use Option 3 instead (existing TCGA clinical data)")
else:
    print(f"\n✓ Found {len(overlap)} overlapping patients!")
    print(f"Sample: {list(overlap)[:5]}")


OPTION 2 IMPLEMENTATION: Create TCGA-CPTAC Mapping File

Note: TCGA and CPTAC are different cohorts with different patient IDs
A mapping file would need to be obtained from CPTAC documentation or
created if the datasets share any overlapping patients.

Checking if there's overlap between TCGA and CPTAC patient IDs...
TCGA IDs sample: ['TCGA-55-6981-01', 'TCGA-86-8280-01', 'TCGA-86-A4P7-01']
CPTAC IDs sample: ['C3L-00510', 'C3N-00547', 'C3N-01021.N']

Patient IDs in common: 0

❌ No overlapping patient IDs found between:
   - TCGA mutation data (TCGA-XX-XXXX format)
   - CPTAC proteomics data (C3L-XXXXX format)

✓ RECOMMENDATION: Use Option 3 instead (existing TCGA clinical data)
